# Chapter 7 PDF Text Extraction and Chucking

## Listing 7.7 Extracting text from PDF

In [41]:
pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.


In [11]:
import PyPDF2
def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        print("Number of PDF pages:", len(reader.pages))
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            text += page_text
        #print(page_text)
    return text

In [12]:
pdf_path='data/test.pdf'

result=extract_text_from_pdf(pdf_path)

Number of PDF pages: 7


### Complete PDF Extraction application

In [16]:
import os
import PyPDF2
from openai import OpenAI
from tqdm import tqdm # for progress bars
import tiktoken as tk
import spacy
from dotenv import load_dotenv

load_dotenv()  # loads from .env automatically

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

# function that extracts text from a PDF
def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        print("Number of PDF pages:", len(reader.pages))
        text = ""
        for  page in reader.pages:
            page_text = page.extract_text()
            text += page_text
            #print(page_text)
    return text

# function that splits the text into chunks based on sentences
def split_sentences_by_spacy(text, max_tokens, 
                        overlap=0, 
                        model="en_core_web_sm"):
    # Load spaCy model
    nlp = spacy.load(model)
    
    # Tokenize the text into sentences using spaCy
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]
    
    # Tokenize sentences into tokens and accumulate tokens
    tokens_lengths = [count_tokens(sent) for sent in sentences]
    
    chunks = []
    start_idx = 0
    
    while start_idx < len(sentences):
        current_chunk = []
        current_token_count = 0
        for idx in range(start_idx, len(sentences)):
            if current_token_count + tokens_lengths[idx] > max_tokens:
                break
            current_chunk.append(sentences[idx])
            current_token_count += tokens_lengths[idx]
        
        chunks.append(" ".join(current_chunk))
        
        # Sliding window adjustment
        if overlap >= len(current_chunk):
            start_idx += 1
        else:
            start_idx += len(current_chunk) - overlap

    return chunks

# count tokens
def count_tokens(string: str, encoding_name="cl100k_base") -> int:
    # Get the encoding
    encoding = tk.get_encoding(encoding_name)
    
    # Encode the string
    encoded_string = encoding.encode(string)

    # Count the number of tokens
    num_tokens = len(encoded_string)
    return num_tokens

# OpenAI embeddings example from Chapter 2
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text)
    
    return response.data[0].embedding

# Process the chunks
def process_chunks(sentences):
    sentence_embeddings = []
    total_token_count = 0

    for sentence in tqdm(sentences):
        # Count the number of tokens in the sentence
        total_token_count += count_tokens(sentence, "cl100k_base")
        
        # Append the sentence and its embedding to the 2D array
        embedding = get_embedding(sentence)
        sentence_embeddings.append([sentence, embedding])

    #print("Simple Sentence Chunking:")
    print("\tNumber of sentence embeddings:", len(sentence_embeddings))
    print("\tTotal number of tokens:", total_token_count)

    return sentence_embeddings

if __name__ == "__main__":
    PDF_PATH = "data/women_fifa_worldcup_2023.pdf"
    extracted_text = extract_text_from_pdf(PDF_PATH)

    # Assuming a chunk size of 2000 characters for demonstration purposes.
    chunks = split_sentences_by_spacy(extracted_text, 2000)
    
    for index, chunk in enumerate(chunks):
        print(f"--- Chunk {index + 1} ---")
        print(chunk)
        print("---------------\n")

Number of PDF pages: 7
--- Chunk 1 ---
FIFA Women's World Cup
Article
Talk
Read
Edit
View history
Tools
From Wikipedia, the free encyclopedia
For the most recent Women's World Cup, see 2023 FIFA Women's World Cup.
 FIFA Women's World Cup
Organising body FIFA
Founded 1991; 32 years ago
Region International
Number of teams 32
Related competitions FIFA World Cup
Current champions  Spain (1st title) (2023)
Most successful team(s)   United States (4 titles)
 Television broadcasters List of broadcasters
Website Official website
Spain, the current champions
Tournaments
1991199519992003200720112015201920232027
The FIFA Women's World Cup is an international association football competition contested by the 
senior women's national teams of the members of Fédération Internationale de Football 
Association (FIFA), the sport's international governing body. The competition has been held 
every four years and one year after the men's FIFA World Cup since 1991, when the inaugural 
tournament, then ca